In [38]:
import pandas as pd
import numpy as np

import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
chrom_list = chrom_list = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 
                           'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 
                           'chr21', 'chr22', 'chrX', 'chrY']

In [5]:
chr_fold_path = "/group/pmc021/amunif/epi-thesis/dataset/chromosome-fold/"

In [58]:
current_chrom = 'chr1'

In [4]:
train_list = list(filter(lambda x: x != current_chrom, chrom_list))
train_list

['chr2',
 'chr3',
 'chr4',
 'chr5',
 'chr6',
 'chr7',
 'chr8',
 'chr9',
 'chr10',
 'chr11',
 'chr12',
 'chr13',
 'chr14',
 'chr15',
 'chr16',
 'chr17',
 'chr18',
 'chr19',
 'chr20',
 'chr21',
 'chr22',
 'chrX',
 'chrY']

In [55]:
all_data = pd.DataFrame()

In [56]:
for chrom in chrom_list:
    print(f"Processing {chrom} ...")
    df = pd.read_csv(f"{chr_fold_path}{chrom}.csv", nrows=3)
    all_data = pd.concat([all_data, df], axis = 0, ignore_index=True)
    del df

Processing chr1 ...
Processing chr2 ...
Processing chr3 ...
Processing chr4 ...
Processing chr5 ...
Processing chr6 ...
Processing chr7 ...
Processing chr8 ...
Processing chr9 ...
Processing chr10 ...
Processing chr11 ...
Processing chr12 ...
Processing chr13 ...
Processing chr14 ...
Processing chr15 ...
Processing chr16 ...
Processing chr17 ...
Processing chr18 ...
Processing chr19 ...
Processing chr20 ...
Processing chr21 ...
Processing chr22 ...
Processing chrX ...
Processing chrY ...


In [57]:
all_data

,h_chrom,h_value_1,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
0,chr1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr1,0.047392,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,chrX,0.016513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,chrX,0.016513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69,chrY,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70,chrY,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [63]:
X_train = all_data[all_data['h_chrom'] != current_chrom]
X_train

,h_chrom,h_value_1,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
3,chr2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,chr2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,chr3,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,chr3,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,chrX,0.016513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,chrX,0.016513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69,chrY,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70,chrY,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
y_train = X_train[['h_value_1']]
y_train

,h_value_1
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
...,...
67,0.016513
68,0.016513
69,0.000000
70,0.000000


In [65]:
X_train = X_train.drop(columns = ['h_chrom', 'h_value_1'])
X_train

,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,h3k4me3_8,h3k4me3_9,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
X_test = all_data[all_data['h_chrom'] == current_chrom]
X_test

,h_chrom,h_value_1,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
0,chr1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr1,0.047392,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
y_test = X_test[['h_value_1']]
y_test

,h_value_1
0,0.000000
1,0.047392
2,0.000000


In [68]:
X_test = X_test.drop(columns=['h_chrom', 'h_value_1'])
X_test

,h3k4me3_0,h3k4me3_1,h3k4me3_2,h3k4me3_3,h3k4me3_4,h3k4me3_5,h3k4me3_6,h3k4me3_7,h3k4me3_8,h3k4me3_9,...,h3k27me3_3990,h3k27me3_3991,h3k27me3_3992,h3k27me3_3993,h3k27me3_3994,h3k27me3_3995,h3k27me3_3996,h3k27me3_3997,h3k27me3_3998,h3k27me3_3999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [70]:
params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.1,
    'seed': 42
}

num_round = 100

In [71]:
bst = xgb.train(params, dtrain, num_round)

In [72]:
# Make predictions
y_pred = bst.predict(dtest)

In [73]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2 Score: {r2}")

MSE: 0.24554731354463247
RMSE: 0.49552730857605864
MAE: 0.4950234252357483
R2 Score: -490.9586181640625


In [53]:
def chrom_fold_xgboost(current_chrom, chrom_list):
    train_list = list(filter(lambda x: x != current_chrom, chrom_list))
    
    all_data = pd.DataFrame()

    for chrom in train_list:
        print(f"Loading data: {chrom} ...")
        df = pd.read_csv(f"{chr_fold_path}{chrom}.csv", nrows=3)
        all_data = pd.concat([all_data, df], axis = 0, ignore_index=True)
        del df

    print("Dataset preparation ...")
    X_train = all_data.drop(columns = ['h_chrom', 'h_value_1'])
    y_train = all_data[['h_value_1']]
    del all_data

    X_test = pd.read_csv(f"{chr_fold_path}{current_chrom}.csv", nrows=3)
    y_test = X_test[['h_value_1']]
    X_test.drop(columns = ['h_chrom', 'h_value_1'], inplace=True)


    print("XGBoost modeling ...")
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'learning_rate': 0.1,
        'seed': 42
    }

    num_round = 50

    bst = xgb.train(params, dtrain, num_round)
    
    print("Making predictions ...")
    # Make predictions
    y_pred = bst.predict(dtest)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"MAE: {mae}")
    print(f"R2 Score: {r2}")

    del X_train
    del y_train
    del X_test
    del y_test